In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
!pip install -q faiss-cpu sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.2 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
import pandas as pd
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [3]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"

train = pd.read_csv(TRAIN_PATH)

print("Train shape:", train.shape)
print(train.head())
print(train.columns.tolist())

Train shape: (2000, 8)
   id                                             prompt  \
0   1  Pick the best possible answer: What is Martin ...   
1   2        What is accelerator-based light-ion fusion?   
2   3  Determine the correct option: What is the term...   
3   4  Select the most accurate option: What is Marti...   
4   5  Identify the correct statement: What is the co...   

                                                   A  \
0  Martin Heidegger believes that humans exist wi...   
1  Accelerator-based light-ion fusion is a techni...   
2                                       Blueshifting   
3  Martin Heidegger believes that humans exist wi...   
4  Simultaneity is relative, meaning that two eve...   

                                                   B  \
0  Martin Heidegger believes that humans do not e...   
1  Accelerator-based light-ion fusion is a techni...   
2                                        Redshifting   
3  Martin Heidegger believes that humans do not e...   

In [4]:
print("Creating knowledge base")

kb = []

for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print("KB size:", len(kb))

print("Loading embedding model and creating index")

model = SentenceTransformer('all-MiniLM-L6-v2')

kb_embeddings = model.encode(
    kb,
    show_progress_bar=True
)

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(np.asarray(kb_embeddings, dtype='float32'))

print("Knowledge base successfully created")
print("FAISS documents:", index.ntotal)

Creating knowledge base
KB size: 2000
Loading embedding model and creating index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge base successfully created
FAISS documents: 2000


In [5]:
device = 0 if torch.cuda.is_available() else -1

zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device
)

row_150 = train.iloc[150]

prompt_150 = str(row_150['prompt'])

labels_150 = [
    str(row_150['A']),
    str(row_150['B']),
    str(row_150['C']),
    str(row_150['D']),
    str(row_150['E'])
]

correct_letter_150 = str(row_150['answer'])
ans_150 = str(row_150[correct_letter_150])

print("Prompt:", prompt_150)
print("Correct letter:", correct_letter_150)
print("Correct option text:", ans_150)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prompt: Select the most accurate option: What is the butterfly effect, as defined by Lorenz in his book "The Essence of Chaos"? based on the given context.
Correct letter: C
Correct option text: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."


In [6]:
result_q1 = zs(
    prompt_150,
    candidate_labels=labels_150
)

print("Labels ranked:")
for label, score in zip(result_q1['labels'], result_q1['scores']):
    print(label, score)

q1_score = result_q1['scores'][
    result_q1['labels'].index(ans_150)
]

print("\nQ1 ANSWER =", round(q1_score, 3))

Labels ranked:
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." 0.38442012667655945
The butterfly effect is the phenomenon that a large change in the initial conditions of a dynamical mechanism can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos." 0.3786771595478058
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical mechanism can cause subsequent states to differ greatly from the states that would have followed without the alteration, as defined by Einstein in his book "The concept of Relativity." 0.09269651025533676
The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical structure has no effect on subsequent states, as defined by Lorenz in his book "The Essence o

In [7]:
prompt_embedding_150 = model.encode(
    [prompt_150],
    show_progress_bar=False
)

distances_10, indices_10 = index.search(
    np.asarray(prompt_embedding_150, dtype='float32'),
    10
)

retrieved_indices = indices_10[0].tolist()

print("Top 10 KB indices:", retrieved_indices)

if 150 in retrieved_indices:
    q2_rank = retrieved_indices.index(150) + 1
else:
    q2_rank = None

print("Q2 ANSWER =", q2_rank)

Top 10 KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
Q2 ANSWER = 10


In [8]:
cross_encoder = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)

docs_10 = [kb[i] for i in retrieved_indices]

pairs = [
    [prompt_150, doc]
    for doc in docs_10
]

ce_scores = cross_encoder.predict(pairs)

rerank_order = np.argsort(ce_scores)[::-1]

reranked_indices = [
    retrieved_indices[i]
    for i in rerank_order
]

print("Original FAISS indices:", retrieved_indices)
print("Reranked KB indices:", reranked_indices)

if 150 in reranked_indices:
    q3_rank = reranked_indices.index(150) + 1
else:
    q3_rank = None

print("Q3 ANSWER =", q3_rank)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Original FAISS indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
Reranked KB indices: [150, 1906, 847, 1693, 1269, 1532, 168, 576, 1701, 663]
Q3 ANSWER = 1


In [9]:
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

prompt_embedding_42 = model.encode(
    [prompt_42],
    show_progress_bar=False
)

distances_5, indices_5 = index.search(
    np.asarray(prompt_embedding_42, dtype='float32'),
    5
)

retrieved_indices_42 = indices_5[0].tolist()

docs_5_42 = [
    kb[i]
    for i in retrieved_indices_42
]

concatenated_docs = " ".join(docs_5_42)

rag_string_42 = (
    f"Context: {concatenated_docs} "
    f"Question: {prompt_42}"
)

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokens = tokenizer(
    rag_string_42,
    truncation=False
)

q4_tokens = len(tokens["input_ids"])

print("Retrieved indices:", retrieved_indices_42)
print("Q4 ANSWER =", q4_tokens)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Retrieved indices: [241, 439, 456, 506, 605]
Q4 ANSWER = 216


In [10]:
true_document_150 = kb[150]

rag_string_150 = (
    f"Context: {true_document_150} "
    f"Question: {prompt_150}"
)

result_q5 = zs(
    rag_string_150,
    candidate_labels=labels_150
)

q5_score = result_q5['scores'][
    result_q5['labels'].index(ans_150)
]

print("Q5 ANSWER =", round(q5_score, 3))

Q5 ANSWER = 0.989


In [11]:
wrong_document = kb[999]

adversarial_rag = (
    f"Context: {wrong_document} "
    f"Question: {prompt_150}"
)

result_q6 = zs(
    adversarial_rag,
    candidate_labels=labels_150
)

q6_score = result_q6['scores'][
    result_q6['labels'].index(ans_150)
]

print("Wrong context:", wrong_document)
print("Q6 ANSWER =", round(q6_score, 3))

Wrong context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.
Q6 ANSWER = 0.529


In [12]:
hits = 0

for i in range(100):
    row = train.iloc[i]

    prompt = str(row['prompt'])
    correct_letter = str(row['answer'])
    correct_option = str(row[correct_letter])

    prompt_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )

    distances, indices = index.search(
        np.asarray(prompt_embedding, dtype='float32'),
        5
    )

    retrieved_docs = [
        kb[j]
        for j in indices[0]
    ]

    is_hit = any(
        correct_option in doc
        for doc in retrieved_docs
    )

    if is_hit:
        hits += 1

hit_rate = (hits / 100) * 100

print("Hits:", hits)
print("Q7 ANSWER =", round(hit_rate, 1))

Hits: 73
Q7 ANSWER = 73.0


In [13]:
map3_scores = []

letters = ['A', 'B', 'C', 'D', 'E']

for i in range(20):
    row = train.iloc[i]

    prompt = str(row['prompt'])
    correct_letter = str(row['answer'])

    option_texts = [
        str(row['A']),
        str(row['B']),
        str(row['C']),
        str(row['D']),
        str(row['E'])
    ]

    # 1. RETRIEVE
    prompt_embedding = model.encode(
        [prompt],
        show_progress_bar=False
    )

    distances, indices = index.search(
        np.asarray(prompt_embedding, dtype='float32'),
        5
    )

    retrieved_indices_i = indices[0].tolist()

    docs_5 = [
        kb[j]
        for j in retrieved_indices_i
    ]

    # 2. RERANK
    pairs = [
        [prompt, doc]
        for doc in docs_5
    ]

    ce_scores = cross_encoder.predict(pairs)

    best_position = int(np.argmax(ce_scores))
    best_document = docs_5[best_position]

    # 3. AUGMENT
    rag_string = (
        f"Context: {best_document} "
        f"Question: {prompt}"
    )

    # 4. PREDICT
    result = zs(
        rag_string,
        candidate_labels=option_texts
    )

    # Map option text back to letters
    label_to_letter = {
        option_texts[j]: letters[j]
        for j in range(5)
    }

    ranked_letters = [
        label_to_letter[label]
        for label in result['labels']
    ]

    top3 = ranked_letters[:3]

    # 5. MAP@3 for one correct answer
    if correct_letter in top3:
        rank = top3.index(correct_letter) + 1
        score = 1.0 / rank
    else:
        score = 0.0

    map3_scores.append(score)

    print(
        f"Row {i:02d} | "
        f"Correct={correct_letter} | "
        f"Top3={top3} | "
        f"Score={score:.3f}"
    )

final_map3 = np.mean(map3_scores)

print("\nQ8 ANSWER =", round(final_map3, 3))

Row 00 | Correct=B | Top3=['B', 'D', 'A'] | Score=1.000
Row 01 | Correct=A | Top3=['A', 'E', 'C'] | Score=1.000
Row 02 | Correct=C | Top3=['C', 'D', 'B'] | Score=1.000
Row 03 | Correct=B | Top3=['B', 'D', 'A'] | Score=1.000
Row 04 | Correct=A | Top3=['A', 'B', 'C'] | Score=1.000
Row 05 | Correct=C | Top3=['B', 'C', 'A'] | Score=0.500
Row 06 | Correct=E | Top3=['E', 'B', 'D'] | Score=1.000
Row 07 | Correct=A | Top3=['A', 'B', 'C'] | Score=1.000
Row 08 | Correct=A | Top3=['A', 'C', 'D'] | Score=1.000
Row 09 | Correct=A | Top3=['A', 'B', 'C'] | Score=1.000
Row 10 | Correct=C | Top3=['C', 'A', 'D'] | Score=1.000
Row 11 | Correct=B | Top3=['B', 'C', 'A'] | Score=1.000
Row 12 | Correct=D | Top3=['D', 'A', 'B'] | Score=1.000
Row 13 | Correct=E | Top3=['E', 'D', 'B'] | Score=1.000
Row 14 | Correct=E | Top3=['E', 'A', 'D'] | Score=1.000
Row 15 | Correct=E | Top3=['E', 'B', 'C'] | Score=1.000
Row 16 | Correct=C | Top3=['C', 'B', 'E'] | Score=1.000
Row 17 | Correct=C | Top3=['C', 'B', 'E'] | Scor